# Fase 6 — Ambiente Gymnasium

Ambiente RL su barre M15. L'ambiente usa il prodotto M15 della Fase 2, feature causali della Fase 3 e l'execution/portfolio engine della Fase 4. L'azione osservata a t viene eseguita sul passaggio a t+1.

**Obiettivi:** definire action/observation spaces, reset/step, episodi train random, validation/test cronologici, protezione out-of-index e verifica con `check_env`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import yaml
from gymnasium.utils.env_checker import check_env

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from gold_rl.data.features import MARKET_FEATURE_COLUMNS, create_market_features
from gold_rl.execution import ExecutionCostsConfig
from gold_rl.trading_env import TradingEnv

config = yaml.safe_load((PROJECT_ROOT / "config.yaml").read_text())
environment = config["environment"]

## 1. M15 + feature causali

Le feature sono ricostruite direttamente da `data/processed/xauusd_m15.parquet` con `strict_windows=True`. Il warmup viene eliminato prima di costruire gli environment.

In [ ]:
m15_path = PROJECT_ROOT / "data/processed/xauusd_m15.parquet"
if not m15_path.exists():
    raise FileNotFoundError("Manca data/processed/xauusd_m15.parquet: eseguire prima la Fase 2.")

ohlcv = pd.read_parquet(m15_path)
if not isinstance(ohlcv.index, pd.DatetimeIndex):
    raise TypeError("Il dataset M15 deve avere un DatetimeIndex.")

features_all = create_market_features(
    ohlcv,
    drop_warmup=True,
    strict_windows=True,
).loc[:, list(MARKET_FEATURE_COLUMNS)]
prices = ohlcv.loc[features_all.index, "Close"].astype("float64")

assert len(features_all) == len(prices) >= 2
assert features_all.index.equals(prices.index)
assert features_all.index.is_monotonic_increasing
assert np.isfinite(features_all.to_numpy()).all()
assert np.isfinite(prices.to_numpy()).all() and (prices > 0).all()

print(f"Dati M15: {len(features_all)}")
print(f"Periodo: {features_all.index.min()} -> {features_all.index.max()}")

## 2. Split cronologico train / validation / test

Per la validazione dell'ambiente i dati vengono separati **prima** di costruire i tre environment, con split temporale 70% / 15% / 15%. Non ci sono sovrapposizioni tra i segmenti.

Il train consente random start e un limite di 5.000 step; validation e test partono dall'inizio del rispettivo segmento e percorrono l'intero periodo.

In [ ]:
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15

n = len(features_all)
train_end = int(n * TRAIN_RATIO)
validation_end = train_end + int(n * VALIDATION_RATIO)

train_features = features_all.iloc[:train_end].copy()
validation_features = features_all.iloc[train_end:validation_end].copy()
test_features = features_all.iloc[validation_end:].copy()

train_prices = prices.loc[train_features.index].copy()
validation_prices = prices.loc[validation_features.index].copy()
test_prices = prices.loc[test_features.index].copy()

assert len(train_features) + len(validation_features) + len(test_features) == n
assert train_features.index.max() < validation_features.index.min()
assert validation_features.index.max() < test_features.index.min()

print(f"Train:      {len(train_features):,} | {train_features.index.min()} -> {train_features.index.max()}")
print(f"Validation: {len(validation_features):,} | {validation_features.index.min()} -> {validation_features.index.max()}")
print(f"Test:       {len(test_features):,} | {test_features.index.min()} -> {test_features.index.max()}")

## 3. Environment e execution engine

L'ambiente riusa `execute_order` e `update_portfolio` della Fase 4. Mapping delle azioni: **0 = short, 1 = flat, 2 = long**.

I costi vengono letti da `config.yaml`. Il reward corrente è il net return dell'equity; i costi di spread, commissione, slippage e turnover penalty sono già inclusi nell'execution engine. `turnover_reward_weight=0.0` significa che non viene applicata una penalizzazione aggiuntiva separata.

In [ ]:
costs = ExecutionCostsConfig(
    spread_rate=float(environment["spread_rate"]),
    commission_rate=float(environment["commission_rate"]),
    slippage_rate=float(environment["slippage_rate"]),
    turnover_penalty=float(environment["turnover_penalty"]),
)

def make_env(features, prices, mode):
    return TradingEnv(
        features=features,
        mid_prices=prices,
        execution_config=costs,
        initial_balance=float(environment["initial_balance"]),
        mode=mode,
        max_episode_steps=5000 if mode == "train" else None,
        random_start=(mode == "train"),
        turnover_reward_weight=0.0,
    )

train_env = make_env(train_features, train_prices, "train")
validation_env = make_env(validation_features, validation_prices, "validation")
test_env = make_env(test_features, test_prices, "test")

assert train_env.action_space.n == 3
assert train_env.random_start is True
assert validation_env.random_start is False
assert test_env.random_start is False

print("Action space:", train_env.action_space)
print("Observation shape:", train_env.observation_space.shape)

## 4. Gymnasium check_env

Il controllo viene eseguito su **tutti e tre** gli environment.

In [ ]:
for name, env in [
    ("TRAIN", train_env),
    ("VALIDATION", validation_env),
    ("TEST", test_env),
]:
    check_env(env, skip_render_check=True)
    print(f"{name} check_env: OK")

## 5. Random agent e protezione out-of-index

Il train viene verificato con un episodio random di 5.000 step. Validation e test vengono percorsi per intero.

Ad ogni step controlliamo che observation, reward ed equity siano finiti. Il test dell'ambiente distingue correttamente `terminated` dalla fine naturale dei dati e `truncated` dal raggiungimento del limite artificiale dell'episodio.

In [ ]:
def run_episode(name, env, full_episode):
    obs, _ = env.reset(seed=42)
    assert np.isfinite(obs).all()

    steps = 0
    while True:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)

        assert np.isfinite(obs).all()
        assert np.isfinite(reward)
        assert np.isfinite(info["equity"])

        steps += 1
        if terminated or truncated:
            break

    if full_episode:
        assert steps == len(env.features) - 1

    print(
        f"{name}: OK | steps={steps:,} | "
        f"final_equity={info['equity']:.2f}"
    )
    return steps, info["equity"]

train_steps, train_equity = run_episode("TRAIN random", train_env, False)
validation_steps, validation_equity = run_episode("VALIDATION full", validation_env, True)
test_steps, test_equity = run_episode("TEST full", test_env, True)

assert train_steps == 5000
assert validation_steps == len(validation_features) - 1
assert test_steps == len(test_features) - 1

## 6. Risultato della validazione

La validazione reale della Fase 6 ha prodotto:

- **M15:** 398.628 osservazioni dopo il warmup.
- **Train:** 279.039 osservazioni; episodio random limitato a 5.000 step.
- **Validation:** 59.794 osservazioni; episodio completo.
- **Test:** 59.795 osservazioni; episodio completo.
- `check_env`: OK su train, validation e test.
- Test suite del progetto: **32 passed**.
- Nessun accesso oltre indice e nessun valore non-finito durante gli episodi.

I due warning di Gymnasium relativi a `Box(-inf, +inf)` sull'observation space non sono failure: `check_env` passa comunque. Restano un possibile miglioramento tecnico futuro.

> Nota: le equity ottenute dal random agent sono risultati di integrazione dell'ambiente, **non** una valutazione della performance della strategia RL.

In [ ]:
print("=== PHASE 6 VALIDATION SUMMARY ===")
print(f"Train observations:      {len(train_features):,}")
print(f"Validation observations: {len(validation_features):,}")
print(f"Test observations:       {len(test_features):,}")
print(f"Train random steps:      {train_steps:,}")
print(f"Validation full steps:   {validation_steps:,}")
print(f"Test full steps:         {test_steps:,}")
print("check_env: TRAIN / VALIDATION / TEST = OK")
print("Project test suite: 32 passed")
print("Phase 6 split/integration check: OK")